# Evaluation correctness tests

Tests 2, 3, 9, 10, 13 from the test plan.
`build_k_shared_dataset` excluded — not used in current experiments.
Run cells in order: Test 10 reuses variables from Test 9.

In [1]:
import numpy as np
import torch
from collections import Counter
from create_datasets import build_loi_dataset, build_preference_dataset, build_steiner_dataset
from evaluate import evaluate

In [ ]:
# Test 2a: qrel content correctness — build_loi_dataset
# 'Who likes X?' must point to a doc that contains X
ds, qrels = build_loi_dataset(n=5, m=3)
corpus = list(ds['corpus'].values())
queries = list(ds['queries'].values())

fails = 0
for i, (q, rel) in enumerate(zip(queries, qrels)):
    item = q.removeprefix('Who likes ').removesuffix('?')
    if item not in corpus[rel[0]]:
        print(f'  FAIL query {i}: {item!r} not in doc {rel[0]}: {corpus[rel[0]]!r}')
        fails += 1

result = 'PASS' if not fails else f'FAIL ({fails})'
print(f'Test 2a (loi qrel content): {result} — {len(qrels)} queries')

In [ ]:
# Test 2b: qrel content correctness — build_preference_dataset
# like    → item in 'likes' portion of liker's doc (before ' dislikes ')
# dislike → item in 'dislikes' portion of disliker's doc (after ' dislikes ')
# neutral → item in both liker's and disliker's docs
ds, qrels, sentiments = build_preference_dataset(n=5, m=3)
corpus = list(ds['corpus'].values())
queries = list(ds['queries'].values())

fails = 0
for i, (q, rel, sent) in enumerate(zip(queries, qrels, sentiments)):
    qtype = sent['type']
    if qtype == 'like':
        item = q.removeprefix('Who likes ').removesuffix('?')
        likes_part = corpus[rel[0]].split(' dislikes ', 1)[0]
        if item not in likes_part:
            print(f'  FAIL query {i} (like): {item!r} not in likes portion of doc {rel[0]}')
            print(f'    doc: {corpus[rel[0]]!r}')
            fails += 1
    elif qtype == 'dislike':
        item = q.removeprefix('Who dislikes ').removesuffix('?')
        dislikes_part = corpus[rel[0]].split(' dislikes ', 1)[1] if ' dislikes ' in corpus[rel[0]] else ''
        if item not in dislikes_part:
            print(f'  FAIL query {i} (dislike): {item!r} not in dislikes portion of doc {rel[0]}')
            print(f'    doc: {corpus[rel[0]]!r}')
            fails += 1
    elif qtype == 'neutral':
        item = q.removeprefix('Who has a preference about ').removesuffix('?')
        for ridx in rel:
            if item not in corpus[ridx]:
                print(f'  FAIL query {i} (neutral): {item!r} not in doc {ridx}')
                fails += 1

result = 'PASS' if not fails else f'FAIL ({fails})'
print(f'Test 2b (preference qrel content): {result} — {len(qrels)} queries')

In [ ]:
# Test 2c: qrel content correctness — build_steiner_dataset
# 'Who likes X and Y?' must point to the one doc containing both X and Y
ds, qrels = build_steiner_dataset(n=7)
corpus = list(ds['corpus'].values())
queries = list(ds['queries'].values())

fails = 0
for i, (q, rel) in enumerate(zip(queries, qrels)):
    body = q.removeprefix('Who likes ').removesuffix('?')
    x, y = body.rsplit(' and ', 1)   # items can contain 'and'; split on the last one
    doc = corpus[rel[0]]
    if x not in doc:
        print(f'  FAIL query {i}: {x!r} not in doc {rel[0]}: {doc!r}')
        fails += 1
    if y not in doc:
        print(f'  FAIL query {i}: {y!r} not in doc {rel[0]}: {doc!r}')
        fails += 1

result = 'PASS' if not fails else f'FAIL ({fails})'
print(f'Test 2c (steiner qrel content): {result} — {len(qrels)} queries')

In [ ]:
# Test 3: sentiments alignment — build_preference_dataset
# sentiments[i] metadata must agree with qrels[i]:
#   like    -> qrels[i] == [liker_idx]
#   dislike -> qrels[i] == [disliker_idx]
#   neutral -> set(qrels[i]) == {liker_idx, disliker_idx}
ds, qrels, sentiments = build_preference_dataset(n=5, m=3)

fails = 0
for i, (rel, sent) in enumerate(zip(qrels, sentiments)):
    li, di = sent['liker_idx'], sent['disliker_idx']
    if li == di:
        print(f'  FAIL query {i}: liker_idx == disliker_idx ({li})')
        fails += 1
        continue
    if sent['type'] == 'like' and rel != [li]:
        print(f'  FAIL query {i} (like): qrels={rel}, expected [{li}]')
        fails += 1
    elif sent['type'] == 'dislike' and rel != [di]:
        print(f'  FAIL query {i} (dislike): qrels={rel}, expected [{di}]')
        fails += 1
    elif sent['type'] == 'neutral' and set(rel) != {li, di}:
        print(f'  FAIL query {i} (neutral): qrels={rel}, expected liker={li} disliker={di}')
        fails += 1

result = 'PASS' if not fails else f'FAIL ({fails})'
print(f'Test 3 (sentiments alignment): {result} — {len(qrels)} entries')

In [ ]:
# Test 9: incremental accumulation consistency
# evaluate() with n_values=[10, 20] must give the same n=20 result as n_values=[20].
# Both runs use seed=42, so the doc shuffle is identical — ranks must match exactly.

rng = np.random.default_rng(0)
n_docs, dim, n_queries = 20, 8, 5
doc_embs = rng.standard_normal((n_docs, dim)).astype(np.float32)
doc_embs /= np.linalg.norm(doc_embs, axis=1, keepdims=True)
qry_embs = rng.standard_normal((n_queries, dim)).astype(np.float32)
qry_embs /= np.linalg.norm(qry_embs, axis=1, keepdims=True)
qrels_synth = [[i] for i in range(n_queries)]  # query i -> doc i, all size-1

print('--- two-pass (n_values=[10, 20]) ---')
res_two = evaluate(doc_embs, qry_embs, qrels_synth, [10, 20], n_queries, ks=[1, 5])
print('--- one-pass (n_values=[20]) ---')
res_one = evaluate(doc_embs, qry_embs, qrels_synth, [20],     n_queries, ks=[1, 5])

fails = 0
for key in res_one[20]:
    a, b = res_two[20][key], res_one[20][key]
    if abs(a - b) > 1e-5:
        print(f'  FAIL {key}: two-pass={a:.6f}  one-pass={b:.6f}')
        fails += 1

result = 'PASS' if not fails else f'FAIL ({fails} metrics differ)'
print(f'Test 9 (incremental accumulation): {result}')
print(f'  n=20 results: {res_one[20]}')

In [ ]:
# Test 10: fixed_rel_size=1 vs None — both code paths must produce identical results.
# Reuses doc_embs, qry_embs, qrels_synth, n_queries from Test 9 above.

print('--- fixed_rel_size=1 ---')
res_fixed = evaluate(doc_embs, qry_embs, qrels_synth, [20], n_queries, ks=[1, 5], fixed_rel_size=1)
print('--- fixed_rel_size=None ---')
res_var   = evaluate(doc_embs, qry_embs, qrels_synth, [20], n_queries, ks=[1, 5], fixed_rel_size=None)

fails = 0
for key in res_fixed[20]:
    a, b = res_fixed[20][key], res_var[20][key]
    if abs(a - b) > 1e-5:
        print(f'  FAIL {key}: fixed={a:.6f}  var={b:.6f}')
        fails += 1

result = 'PASS' if not fails else f'FAIL ({fails} metrics differ)'
print(f'Test 10 (fixed vs var path): {result}')
print(f'  fixed: {res_fixed[20]}')
print(f'    var: {res_var[20]}')

In [ ]:
# Test 13: preference dataset item uniqueness
# Each item appears exactly once as liked and once as disliked across all people.
# No person both likes and dislikes the same item.

ds, qrels, sentiments = build_preference_dataset(n=10, m=4)
queries = list(ds['queries'].values())

like_items, dislike_items = [], []
person_liked, person_disliked = {}, {}

for q, sent in zip(queries, sentiments):
    li, di = sent['liker_idx'], sent['disliker_idx']
    if sent['type'] == 'like':
        item = q.removeprefix('Who likes ').removesuffix('?')
        like_items.append(item)
        person_liked.setdefault(li, []).append(item)
    elif sent['type'] == 'dislike':
        item = q.removeprefix('Who dislikes ').removesuffix('?')
        dislike_items.append(item)
        person_disliked.setdefault(di, []).append(item)

fails = 0

for item, cnt in Counter(like_items).items():
    if cnt != 1:
        print(f'  FAIL: {item!r} liked {cnt} times (expected 1)')
        fails += 1

for item, cnt in Counter(dislike_items).items():
    if cnt != 1:
        print(f'  FAIL: {item!r} disliked {cnt} times (expected 1)')
        fails += 1

if set(like_items) != set(dislike_items):
    only_liked    = set(like_items)    - set(dislike_items)
    only_disliked = set(dislike_items) - set(like_items)
    print(f'  FAIL: only liked={only_liked}')
    print(f'        only disliked={only_disliked}')
    fails += 1

for pidx in set(person_liked) | set(person_disliked):
    overlap = set(person_liked.get(pidx, [])) & set(person_disliked.get(pidx, []))
    if overlap:
        print(f'  FAIL: person {pidx} both likes and dislikes {overlap}')
        fails += 1

result = 'PASS' if not fails else f'FAIL ({fails})'
print(f'Test 13 (item uniqueness): {result}')
print(f'  {len(like_items)} liked, {len(dislike_items)} disliked, {len(set(like_items))} unique items')